In [ ]:
import os, glob, json, re, hashlib
import numpy as np, pandas as pd, joblib
from google.colab import drive
drive.mount("/content/drive")

hits = glob.glob("/content/drive/MyDrive/**/data/features.parquet", recursive=True)
print("found:", hits)
FEAT = hits[0]                                        # the NEW features file (after the rebuild)
ROOT = FEAT.rsplit("/data/", 1)[0]
F = pd.read_parquet(FEAT)
P = pd.DataFrame([json.loads(l) for l in open(f"{ROOT}/data/passages.jsonl")])   # the NEW passages file
P = P[P["context_id"].notna()][["context_id", "rank", "passage", "question"]]
D = F.merge(P, on=["context_id", "rank"], how="left", validate="one_to_one")
assert D["passage"].notna().all() and len(D) == len(F)

SENT_END = re.compile(r"[.!?][\"'\)\]]*$")
D["n_words"] = D["passage"].str.split().str.len()
D["starts_q"] = [float(p.startswith(q + ".")) for p, q in zip(D["passage"], D["question"])]
D["full_stop"] = [float(bool(p.split()) and bool(SENT_END.search(p.split()[-1]))) for p in D["passage"]]

# ---- check 1: the rebuild changed nothing except the new rows
B = pd.read_parquet(f"{ROOT}/data/features_before.parquet")
M = B.merge(F, on=["context_id", "rank"], suffixes=("_b", "_n"), validate="one_to_one")
assert len(M) == len(B), "some rows from the old build are missing in the new one"
print("max abs feature change on the old rows:", {f: float(np.abs(M[f + "_b"] - M[f + "_n"]).max()) for f in ("f1", "f2", "f3")})
print("new rows:", len(F) - len(B), "| injection rows:", int((F["variant"] == "injection").sum()))

# ---- load the frozen scorer and thresholds; do NOT refit or overwrite them
fr = f"{ROOT}/data/frozen"
m = joblib.load(f"{fr}/feature_scorer.joblib")
T = json.load(open(f"{fr}/thresholds.json"))
print("thresholds.json sha256:", hashlib.sha256(open(f"{fr}/thresholds.json", "rb").read()).hexdigest()[:16], "(compare with the committed file)")
D["s_featscorer"] = m["lr"].decision_function(m["scaler"].transform(D[["f1", "f2", "f3"]]))
D["s_length"] = T["directions"]["length"] * D["n_words"]
D["s_retrieval"] = T["directions"]["retrieval_score"] * D["f1"]

D[["dataset", "qid", "split", "context_id", "n_poison", "group_variant", "variant", "source", "rank", "label",
   "n_words", "starts_q", "full_stop", "s_length", "s_retrieval", "s_featscorer"]].to_parquet(f"{ROOT}/data/scores.parquet", index=False)
print("saved scores.parquet:", len(D), "rows | injection rows scored:", int(D[D.variant == "injection"]["s_featscorer"].notna().sum()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
found: ['/content/drive/MyDrive/rag_trust/data/features.parquet']
max abs feature change on the old rows: {'f1': 0.0, 'f2': 0.0, 'f3': 0.0}
new rows: 3000 | injection rows: 20
thresholds.json sha256: 8f4cf6e1c3f7d3c4 (compare with the committed file)
saved scores.parquet: 28650 rows | injection rows scored: 20


In [ ]:
import json
print(json.load(open(f"{ROOT}/data/build_report.json"))["inputs"])

{'poisonedrag_commit': 'f660d72174f06b13fae5163ce656e7b235db858f', 'injection_passages': 20, 'rewritten_passages': 580}


In [ ]:
P = pd.read_json(f"{ROOT}/data/passages_before.jsonl", lines=True)

In [ ]:
   import pandas as pd
   P = pd.read_json(f"{ROOT}/data/passages.jsonl", lines=True)
   print(len(P), "rows")
   print(P["variant"].value_counts().reindex(["injection", "rewritten_clean"]))

32252 rows
variant
injection           20
rewritten_clean    580
Name: count, dtype: int64
